# Modeling: Train Baseline Models


In [1]:
import pandas as pd
import numpy as np
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('../../data/processed/churn_prediction_dataset.csv')
print(f'Loaded dataset: {df.shape}')


Loaded dataset: (78325, 189)


In [2]:
from sklearn.model_selection import train_test_split

# Drop highly correlated columns to reduce multicollinearity
def get_redundant_pairs(df):
    pairs_to_drop = set()
    cols = df.columns
    for i in range(0, df.shape[1]):
        for j in range(0, i+1):
            pairs_to_drop.add((cols[i], cols[j]))
    return pairs_to_drop

def get_top_abs_correlations(df, threshold=0.85):
    au_corr = df.corr().abs().unstack()
    labels_to_drop = get_redundant_pairs(df)
    au_corr = au_corr.drop(labels=labels_to_drop).sort_values(ascending=False)
    return au_corr[au_corr > threshold]

numeric_df = df.select_dtypes(include=[np.number, 'bool'])
high_corr = get_top_abs_correlations(numeric_df, threshold=0.85)

cols_to_drop = set()
for (col1, col2), val in high_corr.items():
    corr1 = abs(df[col1].corr(df['prospect_outcome'])) if col1 != 'prospect_outcome' else 1
    corr2 = abs(df[col2].corr(df['prospect_outcome'])) if col2 != 'prospect_outcome' else 1
    if col1 != 'prospect_outcome' and col2 != 'prospect_outcome':
        if corr1 > corr2:
            cols_to_drop.add(col2)
        else:
            cols_to_drop.add(col1)

print(f'\nDropping {len(cols_to_drop)} redundant features: {list(cols_to_drop)[:5]}...')
df_selected = df.drop(columns=list(cols_to_drop))

# Isolate Target & Strip out Data Leakage
leakage_cols = [
    'prospect_outcome', 'status_scores',
    'total_renewal_score_new', 'membership_renewal_decision',
    'desire_to_cancel_Renewed', 'proforma_membership_status',
    'score_gap'
]
leakage_cols += [c for c in df_selected.columns if 'payment' in c.lower() or 'pricing_tier' in c.lower()]

X = df_selected.drop(columns=[c for c in leakage_cols if c in df_selected.columns])
y = df['prospect_outcome']

print(f'\nFinal Feature Set: {X.shape[1]} features')
X = X.astype(float)


Dropping 56 redundant features: ['serious_complaint', 'crm_dissatisfaction_with_support_Unknown', 'competitor_benefits_mentioned_Unknown', 'starting_membership_net', 'last_total_net_paid']...

Final Feature Set: 120 features


In [3]:
# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Flip target temporarily so 1 = Churn (Intuitive for Recall/Precision measuring)
y_train_churn = (y_train == 0).astype(int)
y_test_churn = (y_test == 0).astype(int)

scale_pos_weight = (y_train_churn == 0).sum() / (y_train_churn == 1).sum()
print(f'Calculated scale_pos_weight for XGBoost: {scale_pos_weight:.2f}')


Calculated scale_pos_weight for XGBoost: 7.37


In [4]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.preprocessing import StandardScaler
import time

# Scale data for LogReg
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

models = {
    'Logistic Regression': LogisticRegression(class_weight='balanced', random_state=42, max_iter=1000),
    'Random Forest': RandomForestClassifier(n_estimators=150, class_weight='balanced', random_state=42, n_jobs=-1, max_depth=10),
    'XGBoost': XGBClassifier(n_estimators=200, scale_pos_weight=scale_pos_weight, random_state=42, max_depth=6, learning_rate=0.05, n_jobs=-1)
}

trained_models = {}
for name, model in models.items():
    print(f'Training {name}... ')
    start_time = time.time()
    X_tr = X_train_scaled if name == 'Logistic Regression' else X_train
    model.fit(X_tr, y_train_churn)
    trained_models[name] = model
    print(f'   Done in {time.time() - start_time:.2f}s')


Training Logistic Regression... 


   Done in 1.07s
Training Random Forest... 


   Done in 3.57s
Training XGBoost... 


   Done in 2.12s


In [5]:
# Save models and data architectures down for Eval and Explainability
os.makedirs('../../models', exist_ok=True)
joblib.dump(trained_models, '../../models/all_trained_models.pkl')
joblib.dump(trained_models['XGBoost'], '../../models/xgb_churn_model.pkl') # Keep purely best model standard format
joblib.dump((X_train, X_test, y_train_churn, y_test_churn, X_train_scaled, X_test_scaled), '../../models/train_test_data.pkl')
print('Saved 3 trained models and caching architectures securely.')


Saved 3 trained models and caching architectures securely.
